# SUMO scenarios (Luxembourg / Ingolstadt / Monaco / Cologne) → TTE format

**Sources, all live, all cloneable without a login:**

| scenario | repo | licence | size |
|---|---|---|---|
| **LuST** (Luxembourg) | `github.com/lcodeca/LuSTScenario` | **MIT** | 449 MB, whole country, 24 h |
| **InTAS** (Ingolstadt) | `github.com/silaslobo/InTAS` | GPL-3.0 | 978 MB, 24 h, validated against 24 count points |
| **MoST** (Monaco) | `github.com/lcodeca/MoSTScenario` | GPLv3 | 46 842 cars + pedestrians + bikes |
| TAPASCologne, Bologna, … | `github.com/DLR-TS/sumo-scenarios` | EPL-2.0 | city → region |

**Read this before using any of it.** The labels here are the output of a
car-following model, not a measurement. LuST and InTAS are calibrated against
*aggregate* counts, which constrains flows and says nothing about whether an
individual trip duration is realistic. Use this as a **diagnostic bench**, never
as a headline benchmark, and never mix simulated and real labels in one training
run without a domain indicator.

**What it buys that no real dataset can:** paired counterfactuals. Re-run the same
demand with an edge closed, with demand at ±20 %, with a different signal plan,
and you get two worlds that differ in exactly one intervention. Nothing in the
real data can do that.

**This conversion is exact, not inferred.** With `--vehroute-output.exit-times`
SUMO writes the edge sequence *and* the exit time of every edge, so
`Timestamps` are the true simulated edge entry/exit times — no map matching, no
segmentation, no guessing.

## Producing the input

```bash
pip install eclipse-sumo sumolib          # brings the binaries with it
git clone https://github.com/lcodeca/LuSTScenario.git

sumo -c LuSTScenario/scenario/dua.static.sumocfg \
     --vehroute-output routes.out.xml \
     --vehroute-output.exit-times true \
     --tripinfo-output tripinfo.xml \
     --time-to-teleport -1                # never teleport: a teleport is a fake route
```

A counterfactual run is the same command with one thing changed, e.g.
`--additional-files close_edge.add.xml` holding a `<closingReroute>`.

## Target format (the "gold" contract)

Taken from the Harbin files in `datasets.zip`, with the Omsk file naming:

| file | contents |
|---|---|
| `matched_trips_<city>.csv` | unnamed index, `Id`, `Coordinates`, `OSMids`, `Timestamps`, `Total_time` |
| `edge_list_directed_<city>.csv` | `osmid_u`, `osmid_v` — directed transitions between road segments |
| `road_network_unique_osmids_<city>.geojson` | one `LineString` per segment, `osmid` / `original_osmid` / `is_duplicate` / `duplicate_index` / `length` / `highway` / ... |

`Coordinates`, `OSMids` and `Timestamps` are Python-literal lists of **equal
length — one entry per GPS fix**: `(lon, lat)` floats, the segment id the fix was
matched to (a string), and the unix timestamp in seconds.
`Total_time = Timestamps[-1] - Timestamps[0]`, in seconds.

In [ ]:
CITY     = "luxembourg"                    # "ingolstadt", "monaco", ...
NET      = "LuSTScenario/scenario/lust.net.xml"
VEHROUTE = "routes.out.xml"                # written by the sumo command above
OUT      = "."

# SUMO counts seconds from 0; anchor them to a wall clock so Timestamps are epochs.
SIM_START = "2020-06-01 00:00:00"

MIN_EDGES   = 3
MIN_SECONDS = 60
MAX_SECONDS = 5400
DROP_DISCONNECTED = True    # a jump to an unreachable edge means the vehicle teleported

In [ ]:
# pip install pandas numpy sumolib eclipse-sumo
import ast, json, os
import numpy as np
import pandas as pd
import sumolib
from tqdm.auto import tqdm

## Helpers

In [ ]:
def load_net(path):
    net = sumolib.net.readNet(path)
    if net.getGeoProj() is None:
        raise ValueError(f"{path} has no projParameter — SUMO cannot give lon/lat for it. "
                         "Use a geo-referenced scenario (LuST, InTAS, MoST, TAPASCologne all are).")
    return net


def edge_lonlat(net, edge):
    """Edge shape in lon/lat."""
    return [tuple(round(c, 6) for c in net.convertXY2LonLat(x, y)) for x, y in edge.getShape()]


def build_edge_list_sumo(net):
    """Successor relation straight out of the network — turn restrictions included."""
    rows = set()
    for e in net.getEdges():
        if e.getID().startswith(":"):
            continue
        for nxt in e.getOutgoing():
            if not nxt.getID().startswith(":"):
                rows.add((e.getID(), nxt.getID()))
    return pd.DataFrame(sorted(rows), columns=["osmid_u", "osmid_v"])


def build_geojson_sumo(net, path):
    feats = []
    for e in net.getEdges():
        eid = e.getID()
        if eid.startswith(":"):
            continue
        shape = edge_lonlat(net, e)
        if len(shape) < 2:
            continue
        props = {"u": e.getFromNode().getID(), "v": e.getToNode().getID(), "key": 0,
                 "osmid": eid, "unique_osmid": eid, "original_osmid": eid,
                 "is_duplicate": False, "duplicate_index": 0,
                 "length": float(e.getLength()),
                 "lanes": str(e.getLaneNumber()),
                 "maxspeed": f"{e.getSpeed() * 3.6:.0f}",
                 "highway": e.getType() or "unclassified",
                 "name": e.getName() or None,
                 "oneway": True}       # SUMO edges are directed by construction
        props = {k: v for k, v in props.items() if v is not None}
        feats.append({"type": "Feature", "properties": props,
                      "geometry": {"type": "LineString", "coordinates": [list(p) for p in shape]}})
    fc = {"type": "FeatureCollection", "name": "road_network_unique_osmids",
          "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}},
          "features": feats}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(fc, f, ensure_ascii=False)
    return len(feats)


def parse_vehroutes(path, net, sim_start_epoch, min_edges=3, min_seconds=60,
                    max_seconds=5400, drop_disconnected=True):
    """`vehroute-output` (with `--vehroute-output.exit-times`) -> gold rows.

    One point per edge entry plus one closing point at the end of the last edge,
    so `Timestamps` are the exact simulated edge entry/exit times.
    """
    shapes, succ = {}, {}
    for e in net.getEdges():
        if e.getID().startswith(":"):
            continue
        shapes[e.getID()] = edge_lonlat(net, e)
        succ[e.getID()] = {n.getID() for n in e.getOutgoing()}

    rows, skipped = [], {"no_exit_times": 0, "short": 0, "teleport": 0, "unknown_edge": 0}
    for veh in tqdm(sumolib.xml.parse(path, "vehicle"), desc="vehicles"):
        route = veh.route[0] if veh.route else None
        if route is None or not getattr(route, "exitTimes", None):
            skipped["no_exit_times"] += 1
            continue

        edges = [e for e in route.edges.split() if not e.startswith(":")]
        times = [float(t) for t in route.exitTimes.split()]
        if len(edges) != len(times):
            skipped["no_exit_times"] += 1
            continue
        if any(e not in shapes for e in edges):
            skipped["unknown_edge"] += 1
            continue
        if len(edges) < min_edges:
            skipped["short"] += 1
            continue
        # a teleport shows up as a jump to an edge the previous one does not reach
        if drop_disconnected and any(b not in succ[a] for a, b in zip(edges, edges[1:])):
            skipped["teleport"] += 1
            continue

        depart = float(veh.depart)
        total = times[-1] - depart
        if not (min_seconds <= total <= max_seconds):
            skipped["short"] += 1
            continue

        entry = [depart] + times[:-1]
        coords = [shapes[e][0] for e in edges] + [shapes[edges[-1]][-1]]
        stamps = [int(round(sim_start_epoch + t)) for t in entry + [times[-1]]]
        osmids = edges + [edges[-1]]

        rows.append({"Id": veh.id,
                     "Coordinates": str(coords),
                     "OSMids": str(osmids),
                     "Timestamps": str(stamps),
                     "Total_time": stamps[-1] - stamps[0]})
    return rows, skipped

## 1. Network

In [ ]:
net = load_net(NET)
edges = [e for e in net.getEdges() if not e.getID().startswith(":")]
print(f"{len(edges)} edges, {len(net.getNodes())} nodes")
print("geo projection:", net.getGeoProj().srs if hasattr(net.getGeoProj(), "srs") else "ok")

## 2. Trips straight out of `vehroute-output`

In [ ]:
sim_start = int(pd.Timestamp(SIM_START, tz="UTC").timestamp())
rows, skipped = parse_vehroutes(VEHROUTE, net, sim_start,
                                min_edges=MIN_EDGES, min_seconds=MIN_SECONDS,
                                max_seconds=MAX_SECONDS,
                                drop_disconnected=DROP_DISCONNECTED)
print(f"kept {len(rows)} trips; skipped {skipped}")

## 3. Write the gold files

In [ ]:
matched = pd.DataFrame(rows, columns=["Id", "Coordinates", "OSMids", "Timestamps", "Total_time"])
matched.to_csv(f"{OUT}/matched_trips_{CITY}.csv", index=True)

edge_list = build_edge_list_sumo(net)
edge_list.to_csv(f"{OUT}/edge_list_directed_{CITY}.csv", index=False)

n_feat = build_geojson_sumo(net, f"{OUT}/road_network_unique_osmids_{CITY}.geojson")
print(len(matched), "trips |", len(edge_list), "transitions |", n_feat, "segments")
matched.head(2)

In [ ]:
def validate_gold(trips_path, edge_list_path=None, geojson_path=None, require_coords=True):
    """Check the produced files against the Harbin/Omsk contract."""
    df = pd.read_csv(trips_path)
    problems = []

    expected = ["Unnamed: 0", "Id", "Coordinates", "OSMids", "Timestamps", "Total_time"]
    if list(df.columns) != expected:
        problems.append(f"columns are {list(df.columns)}, expected {expected}")

    bad_len = bad_eval = bad_total = bad_coord = 0
    osmids_seen = set()
    for _, r in df.iterrows():
        try:
            c = ast.literal_eval(r["Coordinates"])
            o = ast.literal_eval(r["OSMids"])
            t = ast.literal_eval(r["Timestamps"])
        except Exception:
            bad_eval += 1
            continue
        if not (len(c) == len(o) == len(t)):
            bad_len += 1
        if t[-1] - t[0] != r["Total_time"]:
            bad_total += 1
        if require_coords and not all(isinstance(p, tuple) and len(p) == 2 for p in c):
            bad_coord += 1
        osmids_seen.update(map(str, o))

    for label, n in [("rows that do not literal_eval", bad_eval),
                     ("rows with unequal list lengths", bad_len),
                     ("rows where Total_time != Timestamps[-1] - Timestamps[0]", bad_total),
                     ("rows with malformed coordinates", bad_coord)]:
        if n:
            problems.append(f"{n} {label}")

    print(f"{trips_path}: {len(df)} trips, {len(osmids_seen)} distinct segments, "
          f"Total_time median {df['Total_time'].median():.0f} s")

    if edge_list_path:
        el = pd.read_csv(edge_list_path, dtype=str)
        if list(el.columns) != ["osmid_u", "osmid_v"]:
            problems.append(f"edge list columns are {list(el.columns)}")
        known = set(el["osmid_u"]) | set(el["osmid_v"])
        missing = osmids_seen - known
        print(f"{edge_list_path}: {len(el)} transitions, "
              f"{len(osmids_seen & known)}/{len(osmids_seen)} trip segments present")
        if missing and len(missing) > 0.05 * max(len(osmids_seen), 1):
            problems.append(f"{len(missing)} trip segments missing from the edge list")

    if geojson_path:
        with open(geojson_path) as f:
            gj = json.load(f)
        keys = {f["properties"].get("unique_osmid", f["properties"]["osmid"])
                for f in gj["features"]}
        print(f"{geojson_path}: {len(gj['features'])} features, {len(keys)} unique ids")
        if not osmids_seen <= keys:
            problems.append(f"{len(osmids_seen - keys)} trip segments missing from the geojson")

    print("\nOK — matches the gold contract" if not problems
          else "\nPROBLEMS:\n  " + "\n  ".join(problems))
    return df

In [ ]:
_ = validate_gold(f"{OUT}/matched_trips_{CITY}.csv",
                  f"{OUT}/edge_list_directed_{CITY}.csv",
                  f"{OUT}/road_network_unique_osmids_{CITY}.geojson")

## 4. Counterfactual pairs

Run the scenario twice — baseline and intervention — and convert both. Two files
with the *same* `Id` set are two worlds that differ in one thing, which is the
whole reason to be here.

In [ ]:
# Convert a second (intervention) run and line the two up by vehicle id.
CF_VEHROUTE = None      # e.g. "routes.closed_edge.out.xml"

if CF_VEHROUTE and os.path.exists(CF_VEHROUTE):
    cf_rows, cf_skipped = parse_vehroutes(CF_VEHROUTE, net, sim_start,
                                          min_edges=MIN_EDGES, min_seconds=MIN_SECONDS,
                                          max_seconds=MAX_SECONDS,
                                          drop_disconnected=DROP_DISCONNECTED)
    cf = pd.DataFrame(cf_rows, columns=["Id", "Coordinates", "OSMids", "Timestamps", "Total_time"])
    cf.to_csv(f"{OUT}/matched_trips_{CITY}_counterfactual.csv", index=True)

    pair = matched[["Id", "Total_time"]].merge(cf[["Id", "Total_time"]], on="Id",
                                               suffixes=("_base", "_cf"))
    pair["delta_s"] = pair.Total_time_cf - pair.Total_time_base
    pair.to_csv(f"{OUT}/counterfactual_pairs_{CITY}.csv", index=False)
    print(len(pair), "paired trips; delta seconds:",
          pair.delta_s.describe()[["mean", "50%", "max"]].round(1).to_dict())
else:
    print("no counterfactual run configured — set CF_VEHROUTE to enable")

## Caveats

* **The label is a model output.** Everything downstream inherits SUMO's
  car-following and lane-changing assumptions. A big enough model will learn the
  simulator, and that is not TTE.
* **Never pool with real data** without a domain flag. This is the single easiest
  way to produce a number that looks good and means nothing.
* **Teleports are fake routes.** SUMO teleports a vehicle that has been stuck too
  long; the vehicle reappears on a disconnected edge and the "route" is a lie.
  Run with `--time-to-teleport -1`; the notebook also drops any route with a
  disconnected pair as a second line of defence (verified: it catches exactly the
  vehicles SUMO logged as teleported).
* **Segment ids are SUMO edge ids, not OSM way ids.** They are stable within a
  scenario and meaningless across scenarios. Do not merge two scenarios' edge
  lists.
* Scenario licences differ — MIT for LuST, GPL for InTAS and MoST, EPL for the
  DLR set. If anything is redistributed, the strictest one wins.
* MATSim scenarios are the same idea via `output_trips.csv` / `output_legs.csv`,
  but matsim.org warns that not every published scenario contains all the files
  needed to run — check each one before planning around it.